## Brain Map Visualization — Cortical Thickness (Prodromal Subgroups)

Companion to `ggseg_thickness_cortical.ipynb`. Compares **RBD** and **Hyposmia**
prodromal subgroups against **Healthy Controls** using the Desikan–Killiany (`dk`) atlas.

**Contrasts:** RBD vs HC · Hyposmia vs HC

**Group N:** RBD = 119 · Hyposmia = 160 · HC = 192

**Input files:** `results/rbd_vs_hc_parcelwise_ttest.csv` · `results/hyposmia_vs_hc_parcelwise_ttest.csv`

**Output dir:** `results/figures/ggseg/subgroups/thickness/`

**Figures produced:**
- `combined_thinning_unthresh_vs_fdr.png` — unthresholded vs FDR-corrected thinning (2 × 2 layout)
- Per-contrast PNGs in subdirectories

In [ ]:
# install.packages(c("tidyverse", "patchwork", "remotes"))
# remotes::install_github("LCBC-UiO/ggseg")
suppressPackageStartupMessages({
  library(ggseg)
  library(tidyverse)
  library(patchwork)
})
cat("ggseg:", as.character(packageVersion("ggseg")), "\n")
cat("ggplot2:", as.character(packageVersion("ggplot2")), "\n")

In [ ]:
RESULTS_DIR <- "../../results"
FIG_DIR     <- file.path(RESULTS_DIR, "figures", "ggseg", "subgroups", "thickness")
dir.create(FIG_DIR, recursive = TRUE, showWarnings = FALSE)

GROUP_N <- list(
  "RBD vs HC"      = c(n1 = 119, n2 = 192),
  "Hyposmia vs HC" = c(n1 = 160, n2 = 192)
)

TTEST_FILES <- list(
  "RBD vs HC"      = file.path(RESULTS_DIR, "rbd_vs_hc_parcelwise_ttest.csv"),
  "Hyposmia vs HC" = file.path(RESULTS_DIR, "hyposmia_vs_hc_parcelwise_ttest.csv")
)

df_thick <- imap_dfr(TTEST_FILES, function(path, cname) {
  read_csv(path, show_col_types = FALSE) %>% mutate(contrast = cname)
}) %>%
  filter(!is.na(pvalue))

cat("Loaded:", nrow(df_thick), "parcel-contrast rows\n")
cat("Contrasts:", paste(unique(df_thick$contrast), collapse = " | "), "\n")

In [ ]:
contrasts <- setNames(as.list(names(GROUP_N)), names(GROUP_N))

hedges_g_from_t <- function(t, n1, n2, df_resid) {
  -t * sqrt(1/n1 + 1/n2) * (1 - 3 / (4 * df_resid - 1))
}

df_stats <- df_thick %>%
  rowwise() %>%
  mutate(g = hedges_g_from_t(
    Tvalue,
    GROUP_N[[contrast]][["n1"]],
    GROUP_N[[contrast]][["n2"]],
    df
  )) %>%
  ungroup() %>%
  mutate(
    hemi   = if_else(hemi == "L", "left", "right"),
    region = str_replace(parcel, ".*_lab-", ""),
    label  = paste0(if_else(hemi == "left", "lh", "rh"), "_", region)
  )

cat("Hedges' g range:", round(range(df_stats$g, na.rm = TRUE), 3), "\n")

In [ ]:
df_stats <- df_stats %>%
  group_by(contrast) %>%
  mutate(p_fdr = p.adjust(pvalue, method = "fdr")) %>%
  ungroup()

df_stats %>%
  group_by(contrast) %>%
  summarise(
    n_sig_fdr   = sum(p_fdr < 0.05, na.rm = TRUE),
    n_sig_uncor = sum(pvalue < 0.05, na.rm = TRUE),
    n_total     = n(),
    g_max_abs   = round(max(abs(g), na.rm = TRUE), 3)
  )

In [ ]:
G_LIMIT <- 0.5

scale_g <- scale_fill_gradient2(
  low      = "#2166AC",
  mid      = "white",
  high     = "#D6604D",
  midpoint = 0,
  limits   = c(-G_LIMIT, G_LIMIT),
  oob      = scales::squish,
  name     = "Hedges' g",
  na.value = "grey85"
)

# Build 4-view atlas: lateral + medial only
dk_4view <- dk()
dk_4view$data[[1]] <- dk_4view$data[[1]] %>% filter(view %in% c("lateral", "medial"))

### Full Hedges' g maps + FDR-masked maps (per contrast)

In [ ]:
options(repr.plot.width = 14, repr.plot.height = 5)

for (cname in names(contrasts)) {

  p_full <- ggplot(df_stats %>% filter(contrast == cname) %>% select(label, g)) +
    geom_brain(atlas = dk_4view, mapping = aes(fill = g), colour = "white",
               position = position_brain("horizontal")) +
    scale_g +
    labs(title = cname, subtitle = "Cortical thickness — Hedges' g (all parcels)") +
    theme_brain2() +
    theme(plot.title      = element_text(hjust = 0.5, size = 13, face = "bold"),
          plot.subtitle   = element_text(hjust = 0.5, size = 10, colour = "grey40"),
          legend.position = "right")

  p_masked <- ggplot(
    df_stats %>% filter(contrast == cname) %>%
      mutate(g_sig = if_else(p_fdr < 0.05, g, NA_real_)) %>% select(label, g_sig)
  ) +
    geom_brain(atlas = dk_4view, mapping = aes(fill = g_sig), colour = "white",
               position = position_brain("horizontal")) +
    scale_g +
    labs(title = cname, subtitle = "Cortical thickness — FDR-masked (q < 0.05)") +
    theme_brain2() +
    theme(plot.title      = element_text(hjust = 0.5, size = 13, face = "bold"),
          plot.subtitle   = element_text(hjust = 0.5, size = 10, colour = "grey40"),
          legend.position = "right")

  combined <- p_full + p_masked + plot_layout(guides = "collect") & theme(legend.position = "right")
  print(combined)

  fname <- tolower(str_replace_all(cname, " ", "_"))
  ggsave(file.path(FIG_DIR, paste0(fname, ".png")), combined, width = 14, height = 5, dpi = 300)
  cat("Saved:", fname, "\n")
}

### Combined figure: unthresholded vs FDR-corrected thinning (2 × 2 layout)

**Layout:**

| | Left (all thinning) | Right (FDR q < 0.05) |
|---|---|---|
| **Row 1 (a, b)** | RBD vs HC — all thinning | RBD vs HC — FDR q < 0.05 |
| **Row 2 (c, d)** | Hyposmia vs HC — all thinning | Hyposmia vs HC — FDR q < 0.05 |

Pink scale: white → pink, |Hedges' g|, 0–max_atrophy.

In [ ]:
FIG_DIR_COMBINED <- file.path(FIG_DIR, "combined_atrophy")
dir.create(FIG_DIR_COMBINED, recursive = TRUE, showWarnings = FALSE)

max_atrophy <- ceiling(max(abs(df_stats$g[df_stats$g < 0]), na.rm = TRUE) * 20) / 20
cat("Atrophy scale upper limit:", max_atrophy, "\n")

scale_pink <- scale_fill_gradient(
  low      = "white",
  high     = "pink",
  limits   = c(0, max_atrophy),
  oob      = scales::squish,
  na.value = "grey90",
  name     = "|Hedges' g|\natrophy"
)

make_panel <- function(cname, fdr_only = FALSE) {
  d     <- df_stats %>% filter(contrast == cname)
  n_sig <- sum(d$p_fdr < 0.05 & d$g < 0, na.rm = TRUE)
  if (fdr_only) {
    plot_data <- d %>%
      mutate(atrophy = if_else(g < 0 & p_fdr < 0.05, abs(g), NA_real_)) %>%
      select(label, atrophy)
    subt <- if (n_sig == 0) "No parcel survives FDR correction (q < 0.05)" else
      paste0("FDR-significant thinning: n = ", n_sig, " parcels (q < 0.05)")
  } else {
    plot_data <- d %>%
      mutate(atrophy = if_else(g < 0, abs(g), NA_real_)) %>%
      select(label, atrophy)
    n_unc <- sum(d$pvalue < 0.05 & d$g < 0, na.rm = TRUE)
    subt  <- paste0(n_unc, " parcels p < 0.05 uncorrected (all thinning)")
  }
  ggplot(plot_data) +
    geom_brain(atlas = dk_4view, mapping = aes(fill = atrophy), colour = "grey70",
               position = position_brain("horizontal")) +
    scale_pink +
    labs(title = cname, subtitle = subt) +
    theme_void() +
    theme(
      plot.title    = element_text(hjust = 0.5, size = 11, face = "bold"),
      plot.subtitle = element_text(hjust = 0.5, size = 8.5, colour = "grey35")
    )
}

plots_unc <- map(names(contrasts), ~ make_panel(.x, fdr_only = FALSE))
plots_fdr <- map(names(contrasts), ~ make_panel(.x, fdr_only = TRUE))

fig_combined <- (
  (plots_unc[[1]] | plots_fdr[[1]]) /
  (plots_unc[[2]] | plots_fdr[[2]])
) +
  plot_layout(guides = "collect") +
  plot_annotation(
    title      = "Cortical Thickness: Unthresholded vs FDR-corrected Thinning (Prodromal Subgroups)",
    subtitle   = "Left (a, c): all thinning (g < 0) | Right (b, d): FDR-corrected (q < 0.05)",
    tag_levels = "a",
    theme = theme(
      plot.title    = element_text(hjust = 0.5, size = 13, face = "bold"),
      plot.subtitle = element_text(hjust = 0.5, size = 9,  colour = "grey40")
    )
  ) &
  theme(legend.position = "right")

options(repr.plot.width = 14, repr.plot.height = 9)
print(fig_combined)

out_path <- file.path(FIG_DIR_COMBINED, "combined_thinning_unthresh_vs_fdr.png")
ggsave(out_path, fig_combined, width = 14, height = 9, dpi = 300)
cat("Saved:", out_path, "\n")

### Diverging pastel Hedges’ g maps — subgroup cortical thickness

**Figure.** Diverging Hedges’ g maps of cortical thickness for prodromal subgroups (RBD vs HC; Hyposmia vs HC). Light pink = thinning (g < 0); light blue = thickening (g > 0); grey = no data. Scale capped at ±0.30. Left: all 68 parcels unthresholded. Right: FDR q < 0.05 parcels only (Hyposmia: 1 parcel — R parahippocampal, q = 0.018, g = −0.393; RBD: no FDR-significant parcels).

In [ ]:
CNAMES_DISPLAY_SUB <- c(
  "RBD vs HC"      = "RBD vs HC",
  "Hyposmia vs HC" = "hyposmia vs HC"
)

G_LIMIT_PASTEL <- 0.30

scale_diverging_pastel <- scale_fill_gradient2(
  low      = "lightpink",
  mid      = "white",
  high     = "#AED6F1",
  midpoint = 0,
  limits   = c(-G_LIMIT_PASTEL, G_LIMIT_PASTEL),
  oob      = scales::squish,
  name     = "Hedges' g",
  na.value = "grey85"
)

if (!exists("dk_4view")) {
  dk_4view <- dk()
  dk_4view$data[[1]] <- dk_4view$data[[1]] %>% filter(view %in% c("lateral", "medial"))
}

make_pastel_sub <- function(cname, fdr_only = FALSE) {
  d <- df_stats %>% filter(contrast == cname)
  if (fdr_only) {
    plot_data <- d %>% mutate(g_plot = if_else(p_fdr < 0.05, g, NA_real_)) %>% select(label, g_plot)
  } else {
    plot_data <- d %>% mutate(g_plot = g) %>% select(label, g_plot)
  }
  ggplot(plot_data) +
    geom_brain(
      atlas    = dk_4view,
      mapping  = aes(fill = g_plot),
      colour   = "grey70",
      position = position_brain("horizontal")
    ) +
    scale_diverging_pastel +
    labs(title = CNAMES_DISPLAY_SUB[[cname]]) +
    theme_void() +
    theme(
      plot.title    = element_text(hjust = 0.5, size = 11, face = "bold",
                                   margin = margin(t = 10, b = 3)),
      legend.position = "right",
      plot.margin   = margin(t = 8, r = 8, b = 12, l = 8)
    )
}

# Unthresholded (all 68 parcels)
options(repr.plot.width = 10, repr.plot.height = 8)

panels_unc_sub <- map(names(contrasts), ~make_pastel_sub(.x, fdr_only = FALSE))

fig_div_sub_unc <- wrap_plots(panels_unc_sub, ncol = 1) +
  plot_layout(guides = "collect") +
  plot_annotation(
    title    = "Cortical Thickness Subgroups: Diverging Hedges' g (unthresholded)",
    subtitle = "Light pink = thinning (g < 0) | Light blue = thickening (g > 0) | Scale +/-0.30",
    tag_levels = "a",
    theme = theme(
      plot.title    = element_text(hjust = 0.5, size = 13, face = "bold"),
      plot.subtitle = element_text(hjust = 0.5, size = 9,  colour = "grey40")
    )
  ) &
  theme(legend.position = "right")

print(fig_div_sub_unc)

out_unc_sub <- file.path(FIG_DIR, "figure_diverging_cortical_thickness_subgroups_unthresh.png")
ggsave(out_unc_sub, fig_div_sub_unc, width = 10, height = 8, dpi = 300)
cat("Saved:", out_unc_sub, "\n")

# FDR-masked
panels_fdr_sub <- map(names(contrasts), ~make_pastel_sub(.x, fdr_only = TRUE))

fig_div_sub_fdr <- wrap_plots(panels_fdr_sub, ncol = 1) +
  plot_layout(guides = "collect") +
  plot_annotation(
    title    = "Cortical Thickness Subgroups: Diverging Hedges' g (FDR q < 0.05)",
    subtitle = "Light pink = thinning (g < 0) | Light blue = thickening (g > 0) | Grey = not significant",
    tag_levels = "a",
    theme = theme(
      plot.title    = element_text(hjust = 0.5, size = 13, face = "bold"),
      plot.subtitle = element_text(hjust = 0.5, size = 9,  colour = "grey40")
    )
  ) &
  theme(legend.position = "right")

print(fig_div_sub_fdr)

out_fdr_sub <- file.path(FIG_DIR, "figure_diverging_cortical_thickness_subgroups_fdr.png")
ggsave(out_fdr_sub, fig_div_sub_fdr, width = 10, height = 8, dpi = 300)
cat("Saved:", out_fdr_sub, "\n")

### Combined 4-row figure: unthresholded and FDR-corrected (a–d)

**Layout:**

| Tag | Row |
|-----|-----|
| **a** | RBD vs HC — unthresholded |
| **b** | Hyposmia vs HC — unthresholded |
| **c** | RBD vs HC — FDR q < 0.05 |
| **d** | Hyposmia vs HC — FDR q < 0.05 |

In [ ]:
options(repr.plot.width = 10, repr.plot.height = 16)

fig_div_sub_combined <- wrap_plots(
  panels_unc_sub[[1]],  # a: RBD unthresholded
  panels_unc_sub[[2]],  # b: Hyposmia unthresholded
  panels_fdr_sub[[1]],  # c: RBD FDR
  panels_fdr_sub[[2]],  # d: Hyposmia FDR
  ncol = 1
) +
  plot_layout(guides = "collect") +
  plot_annotation(
    title      = "Cortical Thickness: Diverging Hedges' g Maps (Prodromal Subgroups)",
    subtitle   = "a–b: all parcels unthresholded | c–d: FDR q < 0.05 parcels only",
    tag_levels = "a",
    theme = theme(
      plot.title    = element_text(hjust = 0.5, size = 13, face = "bold"),
      plot.subtitle = element_text(hjust = 0.5, size = 9,  colour = "grey40")
    )
  ) &
  theme(legend.position = "right")

print(fig_div_sub_combined)

out_combined_sub <- file.path(FIG_DIR, "figure_diverging_cortical_thickness_subgroups_combined.png")
ggsave(out_combined_sub, fig_div_sub_combined, width = 10, height = 16, dpi = 300)
cat("Saved:", out_combined_sub, "\n")

### Results table

In [ ]:
df_stats %>%
  select(contrast, hemi, region, g, Tvalue, pvalue, p_fdr) %>%
  mutate(
    across(where(is.numeric), \(x) round(x, 4)),
    sig = if_else(p_fdr < 0.05, "*", "")
  ) %>%
  arrange(contrast, desc(abs(g)))